# Week 01 — Data plumbing and a CVR baseline

**Goal.** Get both Criteo datasets loading, profile them honestly, and establish the baseline that the next eleven weeks are measured against.

**Deliverable.** Repo scaffold, a logistic-regression and a LightGBM CVR baseline, metrics table in the README.

**Rough shape of the week.** 2h reading (FTRL) · 5h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## 1. Profile before you model

You cannot make a good modelling decision about a dataset you have not looked at. Answer
these in the cells below and write the answers into the README — several of them will
change what you do in weeks 4, 6 and 11.

- How many rows, users, campaigns? Over what time window?
- What is the conversion base rate? The click rate?
- What are the cardinalities of `cat1..cat9`? Which are high enough that one-hot is
  impossible?
- What fraction of impressions belong to users with more than one impression? (That is
  your Week 6 and 11 population.)
- How does the conversion rate drift across the 30 days? Is the last day comparable to
  the first?

Note the trap documented in `data.load_attribution`: `conversion` is an *impression-level*
label meaning "this user converted within 30 days", not "this impression caused it".

In [ ]:
df = data.add_attribution_derived(data.load_attribution())
print(f"rows={len(df):,}  users={df.uid.nunique():,}  campaigns={df.campaign.nunique():,}")
print(f"conversion={df.conversion.mean():.4%}  click={df.click.mean():.4%}")

df.head()

### Cardinality and sparsity

The number that matters for Week 1 is *distinct values per categorical column*.

Expect a surprise: the `cat*` values are enormous integers, but their distinct counts are
small — around 59k across all nine columns, and only `cat7` is above 2k. **One-hot
encoding is entirely feasible on this dataset.** Do it, and treat the hashed version as
the comparison. Hashing is what production uses because the vocabulary is unknown and
drifting, not because 59k values are too many — and here you can measure exactly what the
collisions cost, which you never can at real scale.

In [ ]:
card = pd.DataFrame({
    "distinct": [df[c].nunique() for c in data.CAT_FEATURES],
}, index=data.CAT_FEATURES)
card["pct_of_rows"] = card.distinct / len(df)
card

### Drift across the window

Plot daily conversion rate and daily volume. If the level moves, a model trained on days
1–21 is already slightly wrong about day 30 before you add any other problem — and that
is the mildest version of the distribution shift the rest of the course is about.

In [ ]:
daily = df.groupby("day").agg(rows=("conversion", "size"), cvr=("conversion", "mean"))

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
ax[0].plot(daily.index, daily.cvr); ax[0].set_ylabel("CVR")
ax[1].bar(daily.index, daily.rows);  ax[1].set_ylabel("impressions"); ax[1].set_xlabel("day")
fig.suptitle("Volume and conversion rate across the window")
print(plots.save(fig, 1, "daily_drift"))

## 2. The split

Everything downstream depends on this being time-ordered. `check_no_leakage` turns that
from a convention into an assertion.

**Worth doing once, for the shock value:** build a random split too, train the same model
on both, and record the AUC gap. That gap is the size of the lie a random split tells you.
Log it as a separate row called `lr_hashed_RANDOM_SPLIT_do_not_trust`.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())
print(f"{len(df):,} rows, {df.timestamp.max()/86400:.1f} days")

sp = split.time_split(df, "timestamp", train_frac=0.7, val_frac=0.1)
split.check_no_leakage(df, sp)
print(sp)

train, val, test = sp.apply(df)

## 3. Baseline 1 — logistic regression on hashed features

The classic. `HashingEncoder` is in the harness; `n_bits` is yours to sweep.

Use `sklearn.linear_model.SGDClassifier(loss="log_loss")` if the full matrix is heavy,
or `LogisticRegression(solver="liblinear")` on a subsample to start.

Look at `enc.collision_report(train)` before you interpret a bad result — at low
`n_bits` you are measuring the hash, not the model.

In [ ]:
from sklearn.linear_model import LogisticRegression

enc = encoders.HashingEncoder(data.CAT_FEATURES, n_bits=18)
print(enc.collision_report(train))

# TODO: fit on train, tune on val, report on test.
# X_train = enc.transform(train); y_train = train.conversion.values
# ...

## 4. Baseline 2 — LightGBM

Trees want the categoricals as `category` dtype, not as hashed sparse columns — give
LightGBM the raw integer codes and let it split on them. That difference in encoding is
itself part of the trees-vs-linear story you will finish in Week 2.

Watch for: LightGBM will happily overfit `uid`. Do not feed it the user id.

In [ ]:
import lightgbm as lgb

feats = data.CAT_FEATURES + ["click_pos", "click_nb", "time_since_last_click", "hour_of_day"]

# TODO: build lgb.Dataset with categorical_feature=data.CAT_FEATURES, early-stop on val.
# Then predict on test and evaluate.

## 5. Compare

Both models, same test set, same metric bundle. Then answer in the README: **which
metric moved and which did not?** If AUC improved but `calibration_ratio` got worse, you
have a better ranker and a worse bidder — and you should be able to say which one the
business wanted.

In [ ]:
# results = {}
# for name, p in [("lr_hashed_2^18", p_lr), ("lightgbm", p_lgb)]:
#     results[name] = metrics.evaluate(test.conversion.values, p)
# pd.DataFrame(results).T

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=1,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=1))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week01_* results/
git commit -m "week 01: <the finding, not the task>"
```